[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Classification Metrics


## What you will be able to do

Score a yes-or-no model by the mistakes it makes of each kind, and choose the probability at which it
says yes from what each kind of mistake costs.

## The idea

### The problem

A road authority wants to know which of its weather stations will have snow on the ground on 1 May,
so that it keeps a plow crew on duty near those and sends the rest home for the summer. A model
trained on 900 of the stations predicts snow or no snow for each one, and on the other 300, which it
has not seen, it is right 84 percent of the time.

That number does not say what the authority needs to know. The model can be wrong in two ways. It can
predict snow at a station that stays clear, and a crew waits for nothing. Or it can predict no snow
at a station that gets it, and a road is left unplowed. Accuracy adds the two together, as if they
cost the same, and to the authority they do not. A crew kept on duty costs a day's wages, and an
unplowed road costs the overtime of calling one in, which here is three times as much.

There is a second thing accuracy hides. The model does not really decide snow or no snow. It
estimates a probability, and `predict` says yes when that probability is above one half. One half
is a convention. A lower bar flags more stations, catching more of the snowy ones and sending more
crews out for nothing, and a higher bar does the opposite.

### What precision and recall are

> For a classifier and one class it can predict, here snow, **precision** is the share of the
> stations it predicted snow for that really had snow, and **recall** is the share of the stations
> that had snow that it predicted. A **confusion matrix** counts the four outcomes, with true classes
> in its rows and predicted classes in its columns. The **threshold** is the probability above which
> a classifier predicts the class.

### Why it works that way

Precision and recall look at the same predictions from opposite ends. Precision starts from what the
model said, and asks how often a yes was right, so false alarms lower it. Recall starts from what
happened, and asks how many of the real cases the model found, so misses lower it. A model that
predicts snow everywhere has a recall of 1.0 and a poor precision, and one that predicts snow at a
single certain station has a precision of 1.0 and almost no recall. Neither number alone tells you
anything. F1, their harmonic mean, sums them up as one number, and like accuracy it weighs the two
kinds of mistake by a rule nobody chose for this problem.

Moving the threshold trades one for the other. That trade has no right answer inside the data. It
depends on what a false alarm and a miss cost, which is a decision for the people who pay for them,
and a model's job is to show them the options.

### Where this shows up

Most classifiers in scikit-learn have `predict_proba`, and those that do not, such as `LinearSVC`,
have `decision_function`, a score a threshold works on in the same way. Every metric here takes the
measured classes and the predicted ones. The same reasoning decides a fraud check, a medical screen
and a spam filter, wherever a miss and a false alarm cost different amounts. The **Imbalanced Classes** notebook
applies it to a class that is rare, where accuracy misleads even more.

### What this notebook covers

- a model of snow on 1 May, and its accuracy on stations it has not seen
- the confusion matrix, and precision, recall and F1 from it
- `classification_report`, which prints them for both classes
- `predict_proba`, and a table of what each threshold flags, finds and misses
- the threshold chosen by the cost of each kind of mistake
- three errors in how the metrics are called

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
from sklearn.metrics import confusion_matrix, precision_score, recall_score

measured = [True, True, True, False, False, False, False, True]
predicted = [True, False, True, True, True, False, False, True]

print(confusion_matrix(measured, predicted))
print(precision_score(measured, predicted), recall_score(measured, predicted))
```

```
[[2 2]
 [1 3]]
0.6 0.75
```

The first row is the four cases that were really `False`, two predicted `False` and two predicted
`True`, the false alarms. The second row is the four that were `True`, with one missed and three
found. Of the five `True` predictions, three were right, a precision of 0.6, and of the four real cases three were
found, a recall of 0.75. Accuracy, five right out of eight, would hide which kind of mistake was made.

## Setup

Two libraries for the data, scikit-learn, and the tools for one model and its metrics.

- `numpy` and `pandas` build the stations and the tables of results, and the cell prints
  scikit-learn's version
- `train_test_split` holds a quarter of the stations back, with the same share of snow in each part
- `cross_val_predict` and `StratifiedKFold` score every training station with a model that did not
  train on it, for choosing the threshold
- `make_pipeline`, `StandardScaler` and `LogisticRegression` are the model, a logistic regression on
  scaled columns, as the **Pipelines** notebook built
- `confusion_matrix`, `accuracy_score`, `precision_score`, `recall_score`, `f1_score` and
  `classification_report` are the metrics

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. The column
this notebook predicts is `snow_on_may_1`, `True` for a station with snow on the ground that day, and
`FEATURES` names the five numeric columns it predicts from.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### A model of snow on 1 May

`stratify=y` keeps the share of snowy stations the same in the training and test sets, which the
**Train and Test** notebook explains:

In [2]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
predicted = model.predict(X_test)

print(len(y_test), "test stations,", int(y_test.sum()), "with snow")
print("accuracy:", round(accuracy_score(y_test, predicted), 3))

300 test stations, 116 with snow
accuracy: 0.84


On 300 stations it has not seen, 116 of them with snow, the model is right 84 percent of the time.

### The confusion matrix

`confusion_matrix` counts the four outcomes. Its rows are what happened and its columns what the model
said, both in the order `False`, `True`, and labeling them makes the table read on its own:

In [3]:
counts = confusion_matrix(y_test, predicted)

pd.DataFrame(counts, index=["clear", "snow"], columns=["predicted clear", "predicted snow"])

,predicted clear,predicted snow
clear,165,19
snow,29,87


165 clear stations were predicted clear and 87 snowy stations were found. The other two cells are the
mistakes: 19 false alarms, clear stations predicted snowy, and 29 misses, snowy stations predicted
clear. The 252 right answers out of 300 are the accuracy of 0.84.

### Precision, recall and F1

In [4]:
print("precision:", round(precision_score(y_test, predicted), 3))
print("recall:   ", round(recall_score(y_test, predicted), 3))
print("f1:       ", round(f1_score(y_test, predicted), 3))

precision: 0.821
recall:    0.75
f1:        0.784


Of the 106 stations predicted to have snow, 87 did, a precision of 0.821. Of the 116 that had snow, the
model found 87, a recall of 0.75. A quarter of the snowy stations would have been sent no crew. Every
metric takes the measured classes first and the predicted ones second, and the scores are for `True`,
the class scikit-learn calls positive by default.

### Every metric for both classes

`classification_report` prints the three for each class, with `support`, the number of stations of
that class in the test set:

In [5]:
print(classification_report(y_test, predicted, digits=3))

              precision    recall  f1-score   support

       False      0.851     0.897     0.873       184
        True      0.821     0.750     0.784       116

    accuracy                          0.840       300
   macro avg      0.836     0.823     0.828       300
weighted avg      0.839     0.840     0.839       300



The `True` row repeats the numbers above. The `False` row asks the same questions of clear stations.
Of those predicted clear, 0.851 were, and of the clear ones, 0.897 were found. `macro avg` averages the
two rows equally, and `weighted avg` weights them by support.

### Probabilities instead of answers

`predict_proba` gives the model's probability of each class, and its second column is the chance of
snow. `predict` says yes where that column is above one half:

In [6]:
snow_chance = model.predict_proba(X_test)[:, 1]

print(snow_chance[:5].round(3))
print("predict is a threshold of 0.5:", ((snow_chance > 0.5) == predicted).all())

[0.001 0.055 0.002 0.012 0.052]
predict is a threshold of 0.5: True


### A table of thresholds from the training stations

A threshold is chosen from data, so it is chosen on the training stations, and the test set is kept
for the check. `cross_val_predict` gives every training station a probability from a model fitted on
the other folds, one that did not train on it:

In [7]:
train_chance = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), X_train, y_train,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]

print(len(train_chance), "training stations, each scored by a model that did not train on it")

900 training stations, each scored by a model that did not train on it


`at_threshold` counts what a threshold flags, finds and misses among those stations, and the loop below
tries nine thresholds. The cost column prices a crew sent for nothing at 1 and a snowy station left
without one at 3, the road authority's costs in units of a day's wages:

In [8]:
def at_threshold(chance, truth, threshold):
    flagged = chance > threshold
    found = int((flagged & truth).sum())
    false_alarms = int((flagged & ~truth).sum())
    missed = int((~flagged & truth).sum())
    return {"threshold": threshold, "flagged": int(flagged.sum()), "found": found,
            "false alarms": false_alarms, "missed": missed,
            "precision": round(found / (found + false_alarms), 3), "recall": round(found / (found + missed), 3),
            "cost": false_alarms * 1 + missed * 3}


steps = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
thresholds = pd.DataFrame([at_threshold(train_chance, y_train, step) for step in steps])
thresholds

,threshold,flagged,found,false alarms,missed,precision,recall,cost
0,0.1,577,341,236,8,0.591,0.977,260
1,0.2,499,327,172,22,0.655,0.937,238
2,0.3,434,309,125,40,0.712,0.885,245
3,0.4,393,299,94,50,0.761,0.857,244
4,0.5,346,282,64,67,0.815,0.808,265
5,0.6,287,248,39,101,0.864,0.711,342
6,0.7,239,214,25,135,0.895,0.613,430
7,0.8,194,183,11,166,0.943,0.524,509
8,0.9,133,126,7,223,0.947,0.361,676


Reading down the table, the threshold rises and fewer stations are flagged. Precision goes from 0.591
at 0.1 to 0.947 at 0.9, and recall falls from 0.977 to 0.361. At 0.5, the threshold `predict` uses,
the cost is 265. From 0.2 to 0.4 the cost is about the same, 238 to 245, and 0.2 is the lowest, with
172 crews sent for nothing and 22 snowy stations missed among the 900. Because a miss costs three
times a false alarm, the cheap thresholds sit below one half.

### Writing the decision down

The chosen threshold becomes part of the model's use. On the test stations, a prediction is the
probability compared with it:

In [9]:
chosen = thresholds.loc[thresholds["cost"].idxmin(), "threshold"]
decided = snow_chance > chosen

print("threshold:", chosen)
print("precision:", round(precision_score(y_test, decided), 3), "| recall:", round(recall_score(y_test, decided), 3))

threshold: 0.2
precision: 0.704 | recall: 0.922


At 0.2, 70 percent of the crews kept on duty at the test stations see snow, and 92 percent of the
snowy stations get one. Other costs move the cheapest row, as the third task below shows, which is why
the people who pay for both kinds of mistake need to see the whole table. The threshold was chosen on
the training stations, each scored by a model that did not train on it, and the test set gives the
check above.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/08-classification-metrics-solutions.ipynb).

**1.** Print the confusion matrix for the threshold of 0.2, as a labeled DataFrame like the worked example's.

In [10]:
# your code here

**2.** Print `classification_report` for the predictions at 0.2, and say which of the snow row's three numbers rose and which fell compared with 0.5.

In [11]:
# your code here

**3.** Price a false alarm at 1 and a miss at 10 instead, and print the threshold from the table with the lowest cost.

In [12]:
# your code here

**4.** Fit a `DecisionTreeClassifier(max_depth=4, random_state=0)` on the same training set, and print its precision and recall on the test set at the default threshold.

In [13]:
# your code here

## Common errors

### ValueError: Classification metrics can't handle a mix of binary and continuous targets

In [14]:
accuracy_score(y_test, snow_chance)

ValueError: Classification metrics can't handle a mix of binary and continuous targets

`snow_chance` holds probabilities, numbers between 0 and 1, and `accuracy_score` compares classes. It
found `True` and `False` on one side and decimals on the other, and stopped. Give it classes, from
`predict` or from a comparison such as `snow_chance >= 0.5`. Probabilities go to metrics built for
them, such as `average_precision_score`, which the **Imbalanced Classes** notebook uses.

### ValueError: pos_label=1 is not a valid label. It should be one of ['clear', 'snow']

In [15]:
named = y.map({True: "snow", False: "clear"})
named_X_train, named_X_test, named_train, named_test = train_test_split(
    X, named, test_size=0.25, random_state=0, stratify=named)
named_model = make_pipeline(StandardScaler(), LogisticRegression()).fit(named_X_train, named_train)

precision_score(named_test, named_model.predict(named_X_test))

ValueError: pos_label=1 is not a valid label. It should be one of ['clear', 'snow']

With classes written as words, scikit-learn cannot tell which one is the positive class. Its default,
`pos_label=1`, matches `True` and 1, and neither word. Name the class the score is about:

In [16]:
print(round(precision_score(named_test, named_model.predict(named_X_test), pos_label="snow"), 3))

0.821


The precision is the 0.821 of the worked example, since the classes are the same with other names.

### No error, and precision and recall swapped

In [17]:
print("precision, arguments swapped:", round(precision_score(predicted, y_test), 3))
print("recall, arguments swapped:   ", round(recall_score(predicted, y_test), 3))

precision, arguments swapped: 0.75
recall, arguments swapped:    0.821


Each metric takes the measured classes first. With the predictions first, precision divides by the
stations that had snow, which is recall's question, so the two numbers trade places, precision
reading 0.75 and recall 0.821. Nothing warns, because both orders are valid input. Write the call with
names, `precision_score(y_true=y_test, y_pred=predicted)`, when the order matters to a reader.

## Recap

- Read a confusion matrix, rows for what happened and columns for what the model said
- Compute precision, recall and F1, and say which kind of mistake lowers each
- Get probabilities from `predict_proba` and compare them with a threshold of your own
- Lay out what each threshold flags, finds and misses, and price the mistakes to choose one
- Give a metric the measured classes first, classes and not probabilities, and the positive class by
  name when the classes are words

## What is next

The **Imbalanced Classes** notebook turns to frost in June, a class only one station in twenty has,
where a model can score 95 percent by never predicting it.

---

&#8592; **Previous:** [Regression Metrics](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/07-regression-metrics.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Imbalanced Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/09-imbalanced-classes.ipynb) &#8594;
